# P-2026-11-16-1 — Does a less pessimistic noise model rescue the "essentially unchanged" band?

**Question** (from `proposals/2026-11-15.md` #1; queue Q01–Q16 are all taken on `main`, and `P-2026-11-13/14/15-1` are taken): the 2026-11-15 run found **no** threshold τ\* under a with-replacement product bootstrap, and said that bootstrap is probably too pessimistic. Tonight we repeat the test under two alternative noise models: (1) an **80% product subsample without replacement** (m-out-of-n), both raw and rescaled, and (2) **multiplicative log-normal noise on each export cell X_cp**, calibrated against the Growth Lab's totals. Does a τ\* exist under either, and what would the Spring 2026 reports print?

- **Date:** 2026-11-16 (nightly run)
- **Data:** `data/raw/2019–2023_mega.csv` → HS4 (reporters ≥ $1B, chapters 98/99 dropped, regional aggregates dropped) via `lib/research.repo_hs`; ECI via `lib/eci.eci_from_exports` (RCA ≥ 1, sorted eigenvalues, sign oriented to diversity). Calibration and benchmark: `external-data/raw/atlas_hs92_country_year.csv` (country totals, ECI) and `external-data/raw/atlas_hs92_product_year_4.csv` (world totals per HS4 product).
- **Years:** 2019 → 2022 primary; 2019 → 2021, 2019 → 2023 and 2021 → 2022 as checks. Calibration uses 2019–2022 only (2023 suspect, D1).
- **Run time:** printed in the last code cell.

## TL;DR

- **The 11-15 bootstrap was not too pessimistic.** The properly rescaled 80% product subsample (V1b, delete-d jackknife) gives a median SE of **0.150 SD** for the 2019 → 2022 change, the same as the with-replacement baseline tonight (**0.151**; 0.159 last night with a different seed). It finds **no τ\*** in any window, and calls only **8** countries "changed" under the z rule. Only the *unrescaled* subsample (V1a) looks better (SE 0.075, τ\* 0.210), and that is half the spread by construction, not a better model.
- **Trade-cell noise gives a τ\*, but its size is set by the noise you assume.** Under V2, τ\* exists in the primary window at every σ tried: **0.025** at the calibrated σ_cal = 0.031, 0.075 at σ = 0.10, 0.110 at σ_sd = 0.196, 0.160 at σ = 0.25 and 0.255 at σ = 0.50. Only σ_cal and σ = 0.10 give a τ\* in all four windows (0.020–0.030 and 0.065–0.105).
- **σ_cal is too small to trust.** It is calibrated on country and product *totals* (robust σ_row 0.027, σ_col 0.016), which average out cell noise. Under it the z rule calls 85 countries, but only **0.694** of those calls agree with the sign of ATLAS's change, and the countries it leaves "unchanged" agree only 0.444 of the time. Raising σ makes the calls fewer and more accurate (ATLAS agreement 0.710 at σ = 0.10 → 0.775 at 0.25 → 0.818 at 0.50 → 1.000 under V0/V1b, with 62 → 40 → 22 → 8–9 calls).
- **For the 7 Spring 2026 countries,** the UK reads "increased" under every model. Poland reads "decreased" under V1a, V2 σ_cal and V2 σ = 0.25, and "unchanged" under V0 and V1b. Germany, Singapore and the UAE read "unchanged" under every model. Canada and Vietnam read "increased" only under σ_cal, and ATLAS's change for Canada is negative (−0.049). Report countries left "unchanged": 6 of 7 (V0, V1b), 5 (V1a, σ = 0.25), 3 (σ_cal).
- **Verdict:** no noise model we can calibrate from repo data gives the lab one defensible constant. The choice is a trade-off between how many calls the reports make and how often those calls agree with ATLAS. The missing piece is a cell-level calibration of σ against the Growth Lab's country × HS4 panel (data request below).

## Why this matters for the lab

The template's `[increased/decreased]` placeholder needs a defensible "essentially unchanged" option, because the 11-14 run showed several Spring 2026 direction calls rest on changes of a few hundredths of an SD. The 11-15 run could not set the cut-off: under its bootstrap, the median SE of a 2019 → 2022 change (0.159 SD) was larger than the median change (0.092 SD). If that bootstrap was the problem, a better noise model should give a usable rule. If no reasonable noise model gives one, the lab should stop looking for a fixed constant and use either a per-country z rule or an ATLAS-validated band. This tests **our implementation** (report wording), not the ECI → growth link.

## Method

**Pre-stated before running any noise draw** (the calibration σ values in Section B were computed during setup, before any bootstrap; the rules and metrics below were fixed before that):

- **Sample:** same as 11-15. Fixed HS4 panel = countries in the HS4 sample in every year 2019–2023, and HS4 products present in every year. Δ_c = ECI_c(2022) − ECI_c(2019) in within-year SD units.
- **Baseline (V0):** the 11-15 paired product bootstrap (draw the product columns with replacement, same draw every year), B = 500, to reproduce last night's numbers.
- **V1a — m-out-of-n subsample, raw:** draw 80% of products **without** replacement (m = ⌊0.8·n⌋), same draw every year, recompute ECI, take Δ_b. Uses the subsample spread as is (the optimistic reading).
- **V1b — m-out-of-n subsample, rescaled (primary for V1):** the delete-d jackknife rescaling. With d = n − m, the full-sample variance ≈ (m/d)·Var(Δ_b), so rescaled draws are Δ + √(m/d)·(Δ_b − mean Δ_b). At 80%, √(m/d) = 2.
- **V2 — trade-noise model (primary for V2):** multiply each positive X_cp by exp(σ·z − σ²/2), z ~ N(0,1), drawn **independently per year** (reporting errors are year-specific). σ = σ_cal = √(σ_row² + σ_col²), where σ_row is the robust SD (1.4826·MAD) of the country-demeaned ln(repo total / ATLAS total) over panel countries and 2019–2022, and σ_col is the same for HS4 world totals vs `atlas_hs92_product_year_4.csv`. **Variants:** σ_sd (the same formula with plain SDs, which keeps the heavy tails) and a fixed grid σ ∈ {0.10, 0.25, 0.50}.
- **Primary metric and rule (unchanged from 11-15):** p_c = share of draws with sign(Δ_b) = sign(Δ_c). τ\* = the smallest τ on 0, 0.005, …, 0.400 such that ≥ 90% of countries with |Δ| ≥ τ have p_c ≥ 0.90.
- **Secondary metrics:** median SE of Δ; share of countries with p ≥ 0.90; per-country z rule (|Δ| ≥ 1.645·SE) call count and its sign agreement with ATLAS ECI changes; the words for the 7 Spring 2026 report countries.
- **Seeds:** 20261116 + variant offset. All variants are reported below, not just the one that works.

In [1]:
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 30)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]
CAL_YEARS = [2019, 2020, 2021, 2022]
REPORT = ['CAN', 'DEU', 'POL', 'SGP', 'ARE', 'GBR', 'VNM']
WINDOWS = [(2019, 2022), (2019, 2021), (2019, 2023), (2021, 2022)]
PRIMARY = (2019, 2022)
B, SEED, PSTAR, SHARE, FRAC = 500, 20261116, 0.90, 0.90, 0.80
GRID = np.round(np.arange(0, 0.4001, 0.005), 3)
print('lib:', eci.__file__)

lib: /home/user/eci-research/willbetts/nightly/runs/2026-11-16_P-2026-11-16-1-noise-models/../../lib/eci.py


### A. Fixed panel and point changes (same construction as 11-15)

In [2]:
X = {y: R.repo_hs(y, 4) for y in YEARS}
panel = sorted(set.intersection(*[set(X[y].index) for y in YEARS]))
prods = sorted(set.intersection(*[set(X[y].columns) for y in YEARS]))
XP = {y: X[y].loc[panel, prods] for y in YEARS}
E = pd.DataFrame({y: eci.eci_from_exports(XP[y]) for y in YEARS})
D = {w: (E[w[1]] - E[w[0]]) for w in WINDOWS}
print('fixed panel:', len(panel), 'countries x', len(prods), 'products; ECI missing cells:', int(E.isna().sum().sum()))
print('report countries missing from panel:', [c for c in REPORT if c not in panel] or 'none')
print('median |delta| 2019->2022 = %.3f SD' % D[PRIMARY].abs().median())

fixed panel: 112 countries x 1222 products; ECI missing cells: 0
report countries missing from panel: none
median |delta| 2019->2022 = 0.092 SD


Takeaway: if the panel and median |Δ| match 11-15 (112 × 1,222, 0.092 SD), tonight's variants are directly comparable to last night's.

### B. Calibrating the trade-noise σ against the Growth Lab's totals

In [3]:
atlas = pd.read_csv('../../external-data/raw/atlas_hs92_country_year.csv')
A = atlas.pivot_table(index='country_iso3_code', columns='year', values='eci')
At = atlas.pivot_table(index='country_iso3_code', columns='year', values='export_value').reindex(panel)[CAL_YEARS]
tot = pd.DataFrame({y: XP[y].sum(axis=1) for y in CAL_YEARS})
Pw = pd.read_csv('../../external-data/raw/atlas_hs92_product_year_4.csv', dtype={'product_hs92_code': str}) \
       .pivot_table(index='product_hs92_code', columns='year', values='export_value')
wt = pd.DataFrame({y: XP[y].sum(axis=0) for y in CAL_YEARS})
common = wt.index.intersection(Pw.index)

def demeaned_logratio(num, den):
    lr = np.log(num / den).replace([np.inf, -np.inf], np.nan).dropna()
    return lr.sub(lr.mean(axis=1), axis=0).stack()
dr = demeaned_logratio(tot, At)
dc = demeaned_logratio(wt.loc[common], Pw.loc[common, CAL_YEARS])
rob = lambda v: 1.4826 * (v - v.median()).abs().median()
sig = {'row_robust': rob(dr), 'col_robust': rob(dc), 'row_sd': dr.std(), 'col_sd': dc.std()}
SIG_CAL = float(np.hypot(sig['row_robust'], sig['col_robust'])); SIG_SD = float(np.hypot(sig['row_sd'], sig['col_sd']))
print('country rows matched to ATLAS: %d of %d; products matched: %d of %d' % (dr.index.get_level_values(0).nunique(), len(panel), len(common), len(prods)))
for k, v in sig.items(): print(f'  sigma_{k:11s} = {v:.4f}')
print('sigma_cal (robust) = %.4f   sigma_sd (plain SD) = %.4f' % (SIG_CAL, SIG_SD))
print('largest |demeaned log ratio| country-years:'); print(dr.abs().sort_values(ascending=False).head(6).round(3).to_string())

country rows matched to ATLAS: 112 of 112; products matched: 1204 of 1222
  sigma_row_robust  = 0.0270
  sigma_col_robust  = 0.0157
  sigma_row_sd      = 0.1232
  sigma_col_sd      = 0.1518
sigma_cal (robust) = 0.0312   sigma_sd (plain SD) = 0.1955
largest |demeaned log ratio| country-years:
QAT  2020    0.963
KWT  2022    0.940
OMN  2022    0.868
QAT  2019    0.776
     2021    0.735
     2022    0.548


Takeaway: σ_cal measures how much our year-specific totals wobble relative to the Growth Lab's cleaned totals. It is computed on **aggregates**, so it is probably a lower bound on cell-level noise; that is why the plain-SD version and the fixed grid are also run.

### C. Running the noise models

In [4]:
Xarr = {y: XP[y].to_numpy(float) for y in YEARS}
def eci_arr(a, cols=None):
    df = pd.DataFrame(a, index=panel, columns=(cols if cols is not None else prods))
    return eci.eci_from_exports(df).reindex(panel)

def run_product(kind, seed):
    rng = np.random.default_rng(seed); P = len(prods); m = int(np.floor(FRAC * P)); out = {y: [] for y in YEARS}
    for b in range(B):
        cols = rng.integers(0, P, P) if kind == 'with_repl' else np.sort(rng.choice(P, m, replace=False))
        for y in YEARS:
            out[y].append(eci_arr(Xarr[y][:, cols], cols=range(len(cols))))
    return {y: pd.concat(out[y], axis=1) for y in YEARS}, m

def run_trade(sigma, seed):
    rng = np.random.default_rng(seed); out = {y: [] for y in YEARS}
    for b in range(B):
        for y in YEARS:
            a = Xarr[y]; noise = np.exp(sigma * rng.standard_normal(a.shape) - sigma ** 2 / 2)
            out[y].append(eci_arr(a * noise))
    return {y: pd.concat(out[y], axis=1) for y in YEARS}

BOOT, TIMES = {}, {}
t = time.time(); BOOT['V0 with-repl (11-15)'], _ = run_product('with_repl', SEED); TIMES['V0'] = time.time() - t
t = time.time(); BSUB, M_SUB = run_product('subsample', SEED + 1); TIMES['V1'] = time.time() - t
SCALE = np.sqrt(M_SUB / (len(prods) - M_SUB))
BOOT['V1a subsample raw'] = BSUB
TRADE = [('V2 trade sigma_cal', SIG_CAL), ('V2 trade sigma_sd', SIG_SD), ('V2 trade sigma=0.10', 0.10),
         ('V2 trade sigma=0.25', 0.25), ('V2 trade sigma=0.50', 0.50)]
for i, (name, s_) in enumerate(TRADE):
    t = time.time(); BOOT[name] = run_trade(s_, SEED + 10 + i); TIMES[name] = time.time() - t
print('m = %d of %d products; jackknife scale sqrt(m/d) = %.3f' % (M_SUB, len(prods), SCALE))
print('seconds per variant:', {k: round(v) for k, v in TIMES.items()})
print('missing country-draws:', {k: int(sum(v[y].isna().sum().sum() for y in YEARS)) for k, v in BOOT.items()})

m = 977 of 1222 products; jackknife scale sqrt(m/d) = 1.997
seconds per variant: {'V0': 60, 'V1': 52, 'V2 trade sigma_cal': 64, 'V2 trade sigma_sd': 65, 'V2 trade sigma=0.10': 66, 'V2 trade sigma=0.25': 67, 'V2 trade sigma=0.50': 66}


missing country-draws: {'V0 with-repl (11-15)': 0, 'V1a subsample raw': 0, 'V2 trade sigma_cal': 0, 'V2 trade sigma_sd': 0, 'V2 trade sigma=0.10': 0, 'V2 trade sigma=0.25': 0, 'V2 trade sigma=0.50': 0}


In [5]:
def draws(name, w):
    if name == 'V1b subsample rescaled':
        db = BSUB[w[1]] - BSUB[w[0]]
        return db.sub(db.mean(axis=1), axis=0).mul(SCALE).add(D[w], axis=0)
    BS = BOOT[name]; return BS[w[1]] - BS[w[0]]
VARIANTS = ['V0 with-repl (11-15)', 'V1a subsample raw', 'V1b subsample rescaled'] + [n for n, _ in TRADE]

def stats_for(name, w):
    d0 = D[w]; db = draws(name, w)
    same = np.sign(db).eq(np.sign(d0), axis=0)
    return pd.DataFrame({'delta': d0, 'abs_delta': d0.abs(), 'se': db.std(axis=1), 'p_sign': same.mean(axis=1),
                         'bias': db.mean(axis=1) - d0})
def tau_star(st):
    for tau in GRID:
        a = st[st.abs_delta >= tau]
        if len(a) and (a.p_sign >= PSTAR).mean() >= SHARE: return float(tau)
    return np.nan
S = {(v, w): stats_for(v, w) for v in VARIANTS for w in WINDOWS}
def atlas_z(st, w):
    ad = (A[w[1]] - A[w[0]]).reindex(st.index); m = ad.notna()
    hit = (np.sign(st.delta) == np.sign(ad))[m]; called = ((st.abs_delta / st.se) >= 1.645)[m]
    return int(called.sum()), hit[called].mean() if called.any() else np.nan, hit[~called].mean() if (~called).any() else np.nan
rows = []
for v in VARIANTS:
    st = S[(v, PRIMARY)]; nz, az, au = atlas_z(st, PRIMARY)
    rows.append({'variant': v, 'median_SE': st.se.median(), 'median_|bias|': st.bias.abs().median(),
                 'share_p>=0.90': (st.p_sign >= PSTAR).mean(), 'tau*': tau_star(st),
                 'z_calls': nz, 'z_atlas_agree': az, 'unchanged_atlas_agree': au})
MAIN = pd.DataFrame(rows).set_index('variant')
print('2019->2022, 112-country fixed HS4 panel; median |delta| = %.3f' % D[PRIMARY].abs().median())
MAIN.round(3)

2019->2022, 112-country fixed HS4 panel; median |delta| = 0.092


,median_SE,median_|bias|,share_p>=0.90,tau*,z_calls,z_atlas_agree,unchanged_atlas_agree
variant,,,,,,,
V0 with-repl (11-15),0.151,0.039,0.107,NaN,9,1.000,0.602
V1a subsample raw,0.075,0.027,0.455,0.210,42,0.762,0.557
V1b subsample rescaled,0.150,0.000,0.161,NaN,8,1.000,0.606
V2 trade sigma_cal,0.027,0.010,0.848,0.025,85,0.694,0.444
V2 trade sigma_sd,0.071,0.031,0.491,0.110,48,0.729,0.562
V2 trade sigma=0.10,0.048,0.022,0.661,0.075,62,0.710,0.540
V2 trade sigma=0.25,0.080,0.030,0.455,0.160,40,0.775,0.556
V2 trade sigma=0.50,0.121,0.047,0.188,0.255,22,0.818,0.589


Takeaway: the headline table. Each row is one noise model; `tau*` = NaN means the pre-stated rule still finds no threshold.

In [6]:
TW = pd.DataFrame({v: {f'{a}->{b}': tau_star(S[(v, (a, b))]) for a, b in WINDOWS} for v in VARIANTS}).T
ZW = pd.DataFrame({v: {f'{a}->{b}': atlas_z(S[(v, (a, b))], (a, b))[0] for a, b in WINDOWS} for v in VARIANTS}).T
print('tau* by window:'); print(TW.to_string()); print('\nz-rule calls (of countries ATLAS also scores) by window:'); print(ZW.to_string())

tau* by window:
                        2019->2022  2019->2021  2019->2023  2021->2022
V0 with-repl (11-15)           NaN         NaN         NaN         NaN
V1a subsample raw            0.210         NaN         NaN         NaN
V1b subsample rescaled         NaN         NaN         NaN         NaN
V2 trade sigma_cal           0.025       0.030       0.020       0.025
V2 trade sigma_sd            0.110       0.145       0.090         NaN
V2 trade sigma=0.10          0.075       0.105       0.065       0.080
V2 trade sigma=0.25          0.160       0.155       0.145         NaN
V2 trade sigma=0.50          0.255         NaN       0.300         NaN

z-rule calls (of countries ATLAS also scores) by window:
                        2019->2022  2019->2021  2019->2023  2021->2022
V0 with-repl (11-15)             9          12          23           7
V1a subsample raw               42          42          53          33
V1b subsample rescaled           8          11          19           6
V2 

Takeaway: whether any τ\* found in the primary window holds up in the other windows.

In [7]:
# How the noise models disagree: do they rank countries' SEs the same way?
SE = pd.DataFrame({v: S[(v, PRIMARY)].se for v in VARIANTS})
print('Spearman correlation of per-country SE (2019->2022) across variants:')
print(SE.corr(method='spearman').round(2).to_string())
print('\nlargest-SE countries under V1b and V2 sigma_cal:')
print(SE[['V0 with-repl (11-15)', 'V1b subsample rescaled', 'V2 trade sigma_cal', 'V2 trade sigma=0.25']].sort_values('V1b subsample rescaled', ascending=False).head(8).round(3).to_string())

Spearman correlation of per-country SE (2019->2022) across variants:
                        V0 with-repl (11-15)  V1a subsample raw  V1b subsample rescaled  V2 trade sigma_cal  V2 trade sigma_sd  V2 trade sigma=0.10  V2 trade sigma=0.25  V2 trade sigma=0.50
V0 with-repl (11-15)                    1.00               0.96                    0.96                0.81               0.89                 0.88                 0.91                 0.92
V1a subsample raw                       0.96               1.00                    1.00                0.83               0.90                 0.90                 0.91                 0.93
V1b subsample rescaled                  0.96               1.00                    1.00                0.83               0.90                 0.90                 0.91                 0.93
V2 trade sigma_cal                      0.81               0.83                    0.83                1.00               0.88                 0.93                 0.88   

Takeaway: if the product-based and trade-based SEs rank countries differently, they capture different noise sources, and a single noise model is not enough.

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, v in zip(axes, ['V0 with-repl (11-15)', 'V1b subsample rescaled', 'V2 trade sigma_cal']):
    st = S[(v, PRIMARY)]
    ax.scatter(st.abs_delta, st.p_sign, s=14, c='#4c72b0', alpha=.7)
    for c in REPORT:
        if c in st.index:
            ax.scatter(st.abs_delta[c], st.p_sign[c], s=40, c='#d62728'); ax.annotate(c, (st.abs_delta[c], st.p_sign[c]), fontsize=8, xytext=(3, -9), textcoords='offset points')
    ax.axhline(PSTAR, color='gray', lw=.6); ax.axvline(0.071, color='gray', ls=':', lw=1, label='0.071 band (11-14)')
    t_ = tau_star(st)
    if np.isfinite(t_): ax.axvline(t_, color='k', ls='--', lw=1, label=f'tau* = {t_}')
    ax.set_xscale('symlog', linthresh=0.05); ax.set_xlim(0, 2); ax.set_title(v); ax.set_xlabel('|ECI change 2019->2022| (SD)')
    ax.legend(fontsize=8, loc='lower right')
axes[0].set_ylabel('P(sign as printed)')
fig.tight_layout(); fig.savefig(FIG / 'sign_confidence_by_noise_model.png', dpi=90); plt.show()

Takeaway: sign confidence against change size under the baseline and the two primary new models.

### D. What the template would print for the 7 Spring 2026 countries

In [9]:
ad = (A[PRIMARY[1]] - A[PRIMARY[0]])
def word(d, thr):
    if not np.isfinite(thr): return 'no rule'
    return 'unchanged' if abs(d) < thr else ('increased' if d > 0 else 'decreased')
rows = []
for c in REPORT:
    r = {'iso3': c, 'delta': D[PRIMARY][c], 'atlas_delta': ad.get(c, np.nan)}
    for v in ['V0 with-repl (11-15)', 'V1a subsample raw', 'V1b subsample rescaled', 'V2 trade sigma_cal', 'V2 trade sigma=0.25']:
        st = S[(v, PRIMARY)]; tag = v.split()[0] + ('' if v.split()[0] != 'V2' else '_' + v.split()[-1])
        r[f'SE {tag}'] = st.se[c]; r[f'p {tag}'] = st.p_sign[c]; r[f'z-word {tag}'] = word(st.delta[c], 1.645 * st.se[c])
    rows.append(r)
WORDS = pd.DataFrame(rows).set_index('iso3')
WORDS.to_csv(FIG / 'report_country_words_2019_2022.csv', float_format='%.4f')
show = WORDS[['delta', 'atlas_delta'] + [c for c in WORDS.columns if c.startswith('p ') or c.startswith('z-word')]]
show.round(3)

,delta,atlas_delta,p V0,z-word V0,p V1a,z-word V1a,p V1b,z-word V1b,p V2_sigma_cal,z-word V2_sigma_cal,p V2_sigma=0.25,z-word V2_sigma=0.25
iso3,,,,,,,,,,,,
CAN,0.081,-0.049,0.598,unchanged,0.810,unchanged,0.766,unchanged,0.996,increased,0.634,unchanged
DEU,-0.026,-0.024,0.740,unchanged,0.808,unchanged,0.646,unchanged,0.972,unchanged,0.860,unchanged
POL,-0.124,-0.030,0.926,unchanged,1.000,decreased,0.938,unchanged,1.000,decreased,0.960,decreased
SGP,0.025,-0.031,0.564,unchanged,0.592,unchanged,0.550,unchanged,0.804,unchanged,0.590,unchanged
ARE,-0.044,-0.015,0.424,unchanged,0.458,unchanged,0.568,unchanged,0.732,unchanged,0.440,unchanged
GBR,0.205,0.088,0.926,increased,1.000,increased,0.972,increased,1.000,increased,0.996,increased
VNM,0.069,0.074,0.788,unchanged,0.906,unchanged,0.704,unchanged,1.000,increased,0.938,unchanged


In [10]:
for col in [c for c in WORDS.columns if c.startswith('z-word')]:
    print(f'{col:26s} report countries "unchanged": {int((WORDS[col] == "unchanged").sum())} of 7')

z-word V0                  report countries "unchanged": 6 of 7
z-word V1a                 report countries "unchanged": 5 of 7
z-word V1b                 report countries "unchanged": 6 of 7
z-word V2_sigma_cal        report countries "unchanged": 3 of 7
z-word V2_sigma=0.25       report countries "unchanged": 5 of 7


### Summary numbers

In [11]:
p = MAIN
summary = {'panel_countries': len(panel), 'products': len(prods), 'B': B, 'm_subsample': M_SUB, 'jackknife_scale': round(float(SCALE), 3),
           'median_abs_delta_2019_2022': round(float(D[PRIMARY].abs().median()), 3),
           'sigma_cal': round(SIG_CAL, 4), 'sigma_sd': round(SIG_SD, 4)}
for v in VARIANTS:
    for k in ['median_SE', 'share_p>=0.90', 'tau*', 'z_calls', 'z_atlas_agree']:
        val = p.loc[v, k]; summary[f'{v} | {k}'] = None if pd.isna(val) else round(float(val), 3)
for k, val in summary.items(): print(f'{k:52s} {val}')
R.save_json({'summary': summary, 'tau_star_by_window': TW.to_dict()}, FIG / 'summary.json')
print('runtime: %.1f s' % (time.time() - T0))

panel_countries                                      112
products                                             1222
B                                                    500
m_subsample                                          977
jackknife_scale                                      1.997
median_abs_delta_2019_2022                           0.092
sigma_cal                                            0.0312
sigma_sd                                             0.1955
V0 with-repl (11-15) | median_SE                     0.151
V0 with-repl (11-15) | share_p>=0.90                 0.107
V0 with-repl (11-15) | tau*                          None
V0 with-repl (11-15) | z_calls                       9.0
V0 with-repl (11-15) | z_atlas_agree                 1.0
V1a subsample raw | median_SE                        0.075
V1a subsample raw | share_p>=0.90                    0.455
V1a subsample raw | tau*                             0.21
V1a subsample raw | z_calls                          42.0
V1a subsa

## Caveats

- **σ_cal is a lower bound on cell noise.** It is calibrated on totals (country rows and product columns), and totals average out cell-level errors. A cell-level calibration needs the Growth Lab country × HS4 file, which is gitignored (98 MB) and not on this machine (`data-requests/2026-11-16-atlas-cell-calibration.md`). The plain-SD version (σ_sd = 0.196) keeps the heavy tails (e.g. Qatar 2020, Kuwait 2022 and Oman 2022 log-ratio deviations of 0.963, 0.940 and 0.868), so it is a rough upper reading from the same data.
- **V2 noise is independent across years and independent across cells.** Real reporting errors are partly persistent (the same reporter misclassifies the same way each year), which would make change noise smaller. They are also correlated within a reporter, which would make it larger for that reporter. Neither is modelled.
- **The product and trade models measure different things.** The product models ask "which products define the product space"; the trade model asks "how noisy are the export values". They rank countries' SEs fairly similarly (Spearman 0.81–0.93 between V0/V1 and V2 variants), but nobody has combined them. A combined model would give larger SEs than either alone.
- **The jackknife rescaling assumes a smooth statistic.** ECI is an eigenvector of a binarised matrix, so it is not smooth. The near-equality of V0 and V1b (0.151 vs 0.150 median SE) is reassuring but not proof.
- **ATLAS is a benchmark, not truth.** Its own 2019 → 2022 changes for the report countries are small (−0.049 to +0.088), so sign disagreement on small changes mixes its noise with ours.
- **Seed differences:** V0 tonight uses seed 20261116, last night's used 20261115. Median SE moved from 0.159 to 0.151 and z-rule calls among ATLAS-matched countries from 7 to 9, which gives a sense of Monte Carlo noise at B = 500.
- 2023 is suspect (D1), so the 2019 → 2023 window is a check only. Saudi Arabia is not in the panel (no 2022 exports in the raw pull).

## Helpful to know

- **Don't use an unrescaled subsample as an SE.** An 80% subsample without replacement has about half the spread of the full-sample estimator (scale √(m/d) = 1.997 here). V1a's apparently good τ\* (0.210) disappears once it is rescaled.
- **The product bootstrap is cheap and reproducible:** about 52–67 s per 500-draw variant over 5 years on the 112 × 1,222 panel. The whole notebook runs in about 7.5 minutes.
- **Our totals track the Growth Lab's closely in most country-years.** The robust year-specific deviation is 2.7% for country totals and 1.6% for product totals (2019–2022). The heavy tail is Gulf oil exporters (Qatar, Kuwait, Oman), the same countries with the least stable ECI.
- **Countries with the widest change SEs** under the rescaled subsample are DR Congo (1.295), Guyana (0.926), Brunei (0.796) and Qatar (0.793), all commodity exporters with few RCA products. Any "essentially unchanged" rule will mostly bite there.
- `figures/report_country_words_2019_2022.csv` holds Δ, SE, p_sign and the z-rule word for every report country under five of the noise models. `figures/summary.json` holds the headline numbers and τ\* by window.

## New research avenues (also in `proposals/2026-11-16.md`)

1. **Cell-level σ calibration** (needs the Growth Lab country × HS4 file locally, see data request). Compute the robust SD of the country-year-demeaned ln(X_repo / X_atlas) per HS4 cell for 2019–2022, rerun V2 at that σ, and check whether its τ\* is stable across windows. This is the one number that decides the band.
2. **Combined noise model.** Apply the paired product subsample (rescaled) and trade noise in the same draw, at σ = 0.10 and σ_sd, and see whether any τ\* survives. This gives the honest total SE for the report sentence.
3. **Choose the band by ATLAS agreement, not by noise.** Pre-state a target (e.g. ≥ 0.80 ATLAS sign agreement among calls) and pick the noise model whose z rule hits it with the most calls, using 2019 → 2021 to choose and 2019 → 2022 to test. Tonight, σ = 0.50 reached 0.818 with 22 calls in 2019 → 2022.
4. **Persistent vs transient trade noise.** Split the country-level log ratio into a fixed country effect and a year-specific part (the latter is σ_row). Rerun V2 with a persistent component to see how much the change SE falls.

## Data issues / data needed

- **New request:** `data-requests/2026-11-16-atlas-cell-calibration.md`. A small committed extract of the Growth Lab country × HS4 export panel (112 panel countries, 2019–2022) would let us calibrate trade noise at the cell level instead of from totals. Without it, the V2 σ is a guess bracketed by 0.031 and 0.196.
- Existing requests still apply: the 2023 re-pull and Saudi Arabia's missing 2022 exports (`data-requests/2026-11-12-2023-repull.md`), and the HS-revision mix (`data-requests/2026-11-13-hs-revision-mix.md`). No new data problems were found in the repo data.